In [ ]:
from transformers import BertTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments, BertForMaskedLM, AutoTokenizer, EarlyStoppingCallback
from datasets import load_dataset
import datasets
import torch
import torch.optim as optim
import re
from nltk.tokenize import sent_tokenize
import nltk
import accelerate
from google_cloud_save import gcs_get_dataset, upload_folder
import os
from data_streamer import build_decade_balanced_stream, DECADES

from google.cloud import storage
from google.oauth2 import service_account
import json

nltk.download('punkt_tab')


In [ ]:
#HYPERPARAMETERS FOR TRAINING

model_name = "emanjavacas/MacBERTh"
experiment_name = "McBERTh-Pretrain-100-Samples-fake-test"

epochs = 3
learning_rate = 0.005
batch_size = 32
logging_steps = 100
warmup_ratio = 0.05
weight_decay = 0.01
max_steps = (1855468//32)*epochs #FIXME dont know what this should be
save_steps = 500
save_strategy = 'steps'
eval_strategy = 'steps'
best_model_metric = 'eval_loss'
callbacks = [EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0001)]

mlm_probability = 0.15

max_input_tokens = 512



max_steps

173949

In [ ]:
#Load dataset and choose the model we want to use
gcs_credentaials = "nlp-research-sp26-8499634f1c62.json"
#load the dataset into a hugginface datasets object
dataset = build_decade_balanced_stream(service_account_path = gcs_credentaials)

In [ ]:
# get all dates from dataset, this function currently does years, will need to update to decades
def get_date_tokens(decades):
    unique_dates = [str(d).removesuffix('s') for d in decades]
    custom_date_tokens = [f"<decade_{d}>" for d in unique_dates]
    return custom_date_tokens

In [ ]:
#create the tokenizer and add custom tokens
#extra_special_tokens tag is for any non-standard special tokens so we'll use it for all the dates

date_tokens = get_date_tokens(DECADES)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'additional_special_tokens' : date_tokens})


In [ ]:
#tokenizer function used to map cleaned samples to tokenizer token ids
def tokenize_data(examples):
    result = tokenizer(examples["text"], max_length=512)
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result



In [ ]:
# Create Data collator for Masked Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=mlm_probability)

train_dataset = dataset
val_dataset = build_decade_balanced_stream(service_account_path = gcs_credentaials, split='valid')


# tokenize datasets
train_dataset = dataset.map(tokenize_data, batch_size=batch_size, batched=True)
val_dataset   = val_dataset.map(tokenize_data,   batch_size=batch_size, batched=True)

In [ ]:
# Load pre-trained model and resize to the custom tokenizer
model = BertForMaskedLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
#set model name and output directory
output_dir = f"Models/{experiment_name}"


#Train using 
# training_args = TrainingArguments(
#     output_dir=output_dir,
#     num_train_epochs=epochs,
#     per_device_train_batch_size=batch_size,
#     per_device_eval_batch_size=batch_size,
#     max_steps= max_steps,
#     #saving model based on eval_loss
#     eval_strategy=eval_strategy,
#     save_strategy=save_strategy,
#     load_best_model_at_end=True,
#     metric_for_best_model=best_model_metric,
#     greater_is_better=False,
#     save_total_limit=2,
#     logging_dir=os.path.join(output_dir, '/logs'),
#     logging_steps=logging_steps,
#     warmup_ratio= warmup_ratio,
#     learning_rate=learning_rate,
#     optim='adamw_torch',
# )

#arguments for vertex
training_args = TrainingArguments(
    output_dir=output_dir,
    #num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    max_steps= max_steps,
    #saving model based on eval_loss
    eval_strategy=eval_strategy,
    eval_steps=save_steps,
    save_strategy=save_strategy,
    load_best_model_at_end=True,
    metric_for_best_model=best_model_metric,
    greater_is_better=False,
    save_total_limit=2,
    logging_dir=f"{output_dir}/logs",
    logging_steps=logging_steps,
    warmup_ratio= warmup_ratio,
    learning_rate=learning_rate,
    optim='adamw_torch',
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=callbacks

)

# Pretrain the model
trainer.train()


#save model
save_path = f"{output_dir}/best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
#model info saving

import json


train_results = trainer.state.log_history

lr = training_args.learning_rate                       
warmup_ratio = training_args.warmup_ratio               
weight_decay = training_args.weight_decay    

train_loss = []
eval_loss= []

for log_entry in trainer.state.log_history:
    # Check for training loss key
    if 'loss' in log_entry.keys():
        train_loss.append(log_entry['loss'])
    # Check for evaluation loss key
    elif 'eval_loss' in log_entry.keys():
        eval_loss.append(log_entry['eval_loss'])

data = {
    "model_name": model_name,
    "epoch": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "learning_rate": lr,
    "logging_steps": training_args.logging_steps,
    "warmup_ratio": warmup_ratio,
    "weight_decay": weight_decay,
    "training_loss": train_loss,
    "eval_loss": eval_loss,
}


# Save to file
save_path = f"{output_dir}/parameters.json"
with open(save_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Saved parameters to {save_path}")

In [ ]:
# This would hypothetically upload the model but the files are too large so it times out
'''
use this instead:
gsutil cp -r ./local-folder gs://your-bucket-name/

#upload model to GCS
local_dir = f"{output_dir}/best"
bucket_name = "project3102-model-bucket"
destination_blob_prefix = f"Training-Tests/{experiment_name}/" # Folder path in GCS

#The api may not be able to upload this much data, it likes to time out
upload_folder(credentials_path=gcs_credentaials, bucket_name=bucket_name, destination_blob_prefix=destination_blob_prefix, local_dir=local_dir)

'''


In [ ]:
credentials = service_account.Credentials.from_service_account_file(gcs_credentaials)

client = storage.Client(credentials=credentials)
bucket = client.bucket("project3102-model-bucket")
blob = bucket.blob(f"Training-Tests/{experiment_name}/parameters.json")
blob.upload_from_string(json.dumps(data, indent=4), content_type="application/json")